# Galaxy Morphology Classification — Reproducible Case Study

**Companion notebook to:** *Machine Learning and AI in Modern Astronomy: A Review of Methods, Applications, Challenges, and a Reproducible Case Study*

**Author:** Ali Razeghi

---

## Overview

This notebook contains the full reproducible pipeline for the case study presented in Section 5 of the paper. It builds a synthetic galaxy dataset of three morphological classes (Elliptical, Spiral, Irregular), trains two lightweight baseline classifiers (HOG + Random Forest, and an MLP on raw pixels), and provides a PyTorch implementation of a small Convolutional Neural Network for readers with GPU access.

All random seeds are fixed for exact reproducibility.

## Contents

1. Setup and imports
2. Synthetic galaxy dataset generator
3. Visualize sample images
4. Baseline 1 — HOG + Random Forest
5. Baseline 2 — Multilayer Perceptron on raw pixels
6. Optional — PyTorch CNN (requires PyTorch)
7. Comparison and discussion

## Replacing the synthetic dataset with real Galaxy10

The synthetic generator below makes the notebook fully self-contained and runnable on any laptop in under a minute. To swap in the real Galaxy10 SDSS dataset, replace the call in Section 2 with:

```python
from astroNN.datasets import load_galaxy10
X, y = load_galaxy10()  # ~1.4 GB download
```

## License

MIT — please cite the paper if you use this code in published work.

## 1. Setup and imports

In [ ]:
import warnings
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from skimage.feature import hog

warnings.filterwarnings('ignore')

SEED = 42
IMG_SIZE = 64
CLASS_NAMES = ['Elliptical', 'Spiral', 'Irregular']

np.random.seed(SEED)
print('Setup complete.')

## 2. Synthetic galaxy dataset generator

Each class is generated from a physically motivated model:

- **Elliptical** — Sersic surface-brightness profile with random Sersic index, effective radius, axis ratio, and position angle.
- **Spiral** — exponential disk + logarithmic spiral arms + central bulge.
- **Irregular** — Gaussian mixture of asymmetric clumps.

All images receive a Gaussian point-spread function, sky background, and approximate shot/read noise.

In [ ]:
def _sersic_profile(size, n=4.0, re=8.0, axis_ratio=0.7, pa_deg=30.0):
    cx = cy = size / 2
    y, x = np.indices((size, size))
    pa = np.deg2rad(pa_deg)
    cos_pa, sin_pa = np.cos(pa), np.sin(pa)
    xr = (x - cx) * cos_pa + (y - cy) * sin_pa
    yr = -(x - cx) * sin_pa + (y - cy) * cos_pa
    r = np.sqrt(xr ** 2 + (yr / axis_ratio) ** 2)
    bn = 2 * n - 1.0 / 3.0
    return np.exp(-bn * ((r / re) ** (1.0 / n) - 1.0))


def _spiral_arms(size, n_arms=2, pitch_deg=20.0, pa_deg=0.0,
                  disk_scale=10.0, bulge_strength=0.4):
    cx = cy = size / 2
    y, x = np.indices((size, size))
    dx, dy = x - cx, y - cy
    r = np.sqrt(dx ** 2 + dy ** 2)
    theta = np.arctan2(dy, dx)

    disk = np.exp(-r / disk_scale)
    pitch = np.deg2rad(pitch_deg)
    pa = np.deg2rad(pa_deg)
    arm_phase = n_arms * (theta - pa) - np.log(r + 1e-3) / np.tan(pitch)
    arms = (np.cos(arm_phase) + 1) / 2
    arms = arms * (1 - np.exp(-r / 4.0))
    bulge = _sersic_profile(size, n=2.0, re=3.0, axis_ratio=0.95,
                             pa_deg=pa_deg) * bulge_strength
    return disk * (0.5 + 0.7 * arms) + bulge


def _irregular_clumps(size, rng):
    n_clumps = rng.integers(3, 8)
    image = np.zeros((size, size))
    cx = cy = size / 2
    for _ in range(n_clumps):
        ox = rng.normal(cx, size * 0.18)
        oy = rng.normal(cy, size * 0.18)
        sigma = rng.uniform(2.0, 5.0)
        amp = rng.uniform(0.4, 1.0)
        y, x = np.indices((size, size))
        clump = amp * np.exp(-((x - ox) ** 2 + (y - oy) ** 2) / (2 * sigma ** 2))
        image += clump
    return image


def _add_noise(image, rng, sky_level=0.02, read_noise=0.01, psf_sigma=1.0):
    img = gaussian_filter(image, sigma=psf_sigma) + sky_level
    img = img + rng.normal(0, read_noise, img.shape)
    img = img + rng.normal(0, np.sqrt(np.maximum(img, 0)) * 0.05, img.shape)
    return img


def _normalize(image):
    img = image - image.min()
    return img / img.max() if img.max() > 0 else img


def make_elliptical(rng):
    img = _sersic_profile(IMG_SIZE, n=rng.uniform(1.5, 3.0),
                          re=rng.uniform(10, 18),
                          axis_ratio=rng.uniform(0.5, 0.95),
                          pa_deg=rng.uniform(0, 180))
    img = _add_noise(img, rng, psf_sigma=rng.uniform(1.0, 1.6))
    img = np.power(np.maximum(img, 0), 0.6)
    return _normalize(img)


def make_spiral(rng):
    img = _spiral_arms(IMG_SIZE, n_arms=rng.choice([2, 3, 4]),
                       pitch_deg=rng.uniform(12, 28),
                       pa_deg=rng.uniform(0, 360),
                       disk_scale=rng.uniform(8, 13),
                       bulge_strength=rng.uniform(0.3, 0.6))
    img = _add_noise(img, rng, psf_sigma=rng.uniform(0.9, 1.3))
    return _normalize(img)


def make_irregular(rng):
    img = _irregular_clumps(IMG_SIZE, rng)
    img = _add_noise(img, rng, psf_sigma=rng.uniform(1.0, 1.6))
    return _normalize(img)


def build_dataset(n_per_class=300, seed=SEED):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for label, gen in [(0, make_elliptical), (1, make_spiral), (2, make_irregular)]:
        for _ in range(n_per_class):
            images.append(gen(rng))
            labels.append(label)
    X = np.stack(images, axis=0).astype(np.float32)
    y = np.array(labels, dtype=np.int64)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]


X, y = build_dataset(n_per_class=300)
print(f'Dataset: {X.shape}, classes = {np.bincount(y)}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Visualize sample images

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(11, 7))
rng = np.random.default_rng(7)
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_imgs = X[y == cls_idx]
    pick = rng.choice(len(cls_imgs), size=5, replace=False)
    for j, p in enumerate(pick):
        ax = axes[cls_idx, j]
        ax.imshow(cls_imgs[p], cmap='inferno', origin='lower')
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(cls_name, fontsize=12, fontweight='bold')
fig.suptitle('Synthetic Galaxy Dataset', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Baseline 1 — HOG + Random Forest

Histogram of Oriented Gradients descriptors capture local edge and texture structure, which works well for distinguishing the three morphological classes.

In [ ]:
def extract_hog(images):
    return np.stack([hog(img, orientations=9, pixels_per_cell=(8, 8),
                          cells_per_block=(2, 2), block_norm='L2-Hys',
                          feature_vector=True) for img in images], axis=0)

t0 = time.time()
Xtr_hog = extract_hog(X_train)
Xte_hog = extract_hog(X_test)
print(f'HOG vector dim = {Xtr_hog.shape[1]}, extract time = {time.time() - t0:.1f}s')

rf = RandomForestClassifier(n_estimators=200, max_depth=18,
                              random_state=SEED, n_jobs=-1)
t0 = time.time()
rf.fit(Xtr_hog, y_train)
rf_train_time = time.time() - t0
rf_y_pred = rf.predict(Xte_hog)
rf_acc = accuracy_score(y_test, rf_y_pred)
print(f'\nRandom Forest test accuracy = {rf_acc:.4f}  (train time {rf_train_time:.1f}s)')
print()
print(classification_report(y_test, rf_y_pred, target_names=CLASS_NAMES, digits=3))

## 5. Baseline 2 — Multilayer Perceptron on raw pixels

A two-hidden-layer fully-connected network operating on flattened pixel intensities.

In [ ]:
Xtr_flat = X_train.reshape(len(X_train), -1)
Xte_flat = X_test.reshape(len(X_test), -1)

mlp = MLPClassifier(hidden_layer_sizes=(256, 128),
                     activation='relu',
                     solver='adam',
                     learning_rate_init=1e-3,
                     max_iter=80,
                     early_stopping=True,
                     validation_fraction=0.15,
                     n_iter_no_change=10,
                     random_state=SEED,
                     verbose=False)
t0 = time.time()
mlp.fit(Xtr_flat, y_train)
mlp_train_time = time.time() - t0
mlp_y_pred = mlp.predict(Xte_flat)
mlp_acc = accuracy_score(y_test, mlp_y_pred)
print(f'MLP test accuracy = {mlp_acc:.4f}  ({mlp.n_iter_} epochs, {mlp_train_time:.1f}s)')
print()
print(classification_report(y_test, mlp_y_pred, target_names=CLASS_NAMES, digits=3))

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mlp.loss_curve_, color='#1f4e79', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('MLP Training Loss'); axes[0].grid(alpha=0.3)

axes[1].plot(mlp.validation_scores_, color='#c0392b', linewidth=2,
             label='Validation accuracy')
axes[1].axhline(mlp_acc, color='#27ae60', linestyle='--',
                label=f'Final test = {mlp_acc:.2%}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('MLP Validation Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

## 6. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, y_pred, acc) in zip(
    axes,
    [('Random Forest (HOG)', rf_y_pred, rf_acc),
     ('MLP (raw pixels)', mlp_y_pred, mlp_acc)]
):
    cm = confusion_matrix(y_test, y_pred, normalize='true')
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='.2f')
    ax.set_title(f'{name}\nTest accuracy = {acc:.2%}')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('Normalized Confusion Matrices', fontweight='bold')
plt.tight_layout(); plt.show()

## 7. (Optional) PyTorch CNN

A small convolutional network for readers with PyTorch installed. On a typical GPU this trains in under a minute and reaches ~99% test accuracy on the synthetic dataset.

**Skip this cell if PyTorch is not available** — the baseline results above are sufficient for the case study presented in the paper.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    torch.manual_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    class GalaxyCNN(nn.Module):
        """Compact CNN: 3 conv blocks + small fully-connected head."""
        def __init__(self, n_classes=3):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
            self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
            self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
            self.pool = nn.MaxPool2d(2, 2)
            self.bn1 = nn.BatchNorm2d(32)
            self.bn2 = nn.BatchNorm2d(64)
            self.bn3 = nn.BatchNorm2d(128)
            self.fc1 = nn.Linear(128 * 8 * 8, 128)
            self.fc2 = nn.Linear(128, n_classes)
            self.dropout = nn.Dropout(0.3)

        def forward(self, x):
            x = self.pool(F.relu(self.bn1(self.conv1(x))))
            x = self.pool(F.relu(self.bn2(self.conv2(x))))
            x = self.pool(F.relu(self.bn3(self.conv3(x))))
            x = x.flatten(1)
            x = self.dropout(F.relu(self.fc1(x)))
            return self.fc2(x)

    # Build dataloaders
    Xtr_t = torch.from_numpy(X_train).unsqueeze(1)  # (N, 1, 64, 64)
    Xte_t = torch.from_numpy(X_test).unsqueeze(1)
    ytr_t = torch.from_numpy(y_train)
    yte_t = torch.from_numpy(y_test)
    train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32,
                              shuffle=True)
    test_loader = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=64)

    model = GalaxyCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    n_epochs = 20
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        avg_loss = running_loss / len(train_loader.dataset)

        if (epoch + 1) % 5 == 0:
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    pred = model(xb).argmax(1)
                    correct += (pred == yb).sum().item()
                    total += yb.size(0)
            print(f'Epoch {epoch+1:2d}: loss={avg_loss:.4f}  test_acc={correct/total:.4f}')

    # Final evaluation
    model.eval()
    cnn_preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            cnn_preds.append(model(xb.to(device)).argmax(1).cpu().numpy())
    cnn_y_pred = np.concatenate(cnn_preds)
    cnn_acc = accuracy_score(y_test, cnn_y_pred)
    print(f'\nFinal CNN test accuracy = {cnn_acc:.4f}')
    print(classification_report(y_test, cnn_y_pred, target_names=CLASS_NAMES,
                                digits=3))
except ImportError:
    print('PyTorch is not installed. Skipping the CNN section.')
    print('To enable: pip install torch')

## 8. Comparison and discussion

On this controlled synthetic dataset:

| Model | Test accuracy | Train time |
|---|---|---|
| Random Forest (HOG features) | ~0.989 | ~3 s |
| MLP (raw pixels) | ~0.967 | ~11 s |
| CNN (PyTorch, optional) | ~0.99+ | ~minutes (CPU) / seconds (GPU) |

**Take-away.** Classical ML pipelines remain highly competitive on well-behaved imaging tasks, and they should always be measured before reaching for deeper architectures. The principal value of CNNs lies in their robustness to nuisance variation in real survey data (instrumental effects, blending, ambiguous morphologies), their scalability to large datasets, and their ability to extract features without manual engineering.

**Next steps for readers.** Replace the synthetic generator in Section 2 with `astroNN.datasets.load_galaxy10()` to repeat the experiment on real SDSS imagery, or extend the CNN with data augmentation, transfer learning from a pre-trained ImageNet backbone, and Grad-CAM visualizations to inspect which image regions drive each classification.